# AI Engineering Interview Prep
## Section: FastAPI

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 511+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (10 Qs)",
    "18. FastAPI (10 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## ⚡ Section 18 — FastAPI

### Q1. What is the difference between `async def` and `def` endpoints in FastAPI?
**A:** FastAPI runs on ASGI (Uvicorn) and is async-first.
- If you declare an endpoint as `async def`, FastAPI runs it directly on the main async event loop. It *must* only contain non-blocking async calls (using `await`). If you put a blocking sync call (like `time.sleep()` or a sync database call) inside `async def`, you block the entire event loop, freezing the API for all other users.
- If you declare it as normal `def`, FastAPI automatically offloads it to an internal thread pool so it runs concurrently without blocking the main event loop.
**Rule:** Use `async def` for async-native operations (fetching HTTP endpoints async, async db drivers). Use `def` for standard sync operations (SQLAlchemy sync calls, heavy CPU processing).

### Q2. How does dependency injection (`Depends`) work in FastAPI?
**A:** Dependency Injection allows you to share logic, database connections, authentication guards, and configurations across multiple endpoints without copying code. You pass a callable wrapped in `Depends(my_function)` as a parameter in your path function. FastAPI automatically executes the dependency before running the endpoint and passes the returned value as an argument. Dependencies can also yield resources (using the generator pattern); FastAPI runs the code before `yield`, passes the resource to the endpoint, and executes the cleanup code after `yield` once the request completes.

### Q3. How does FastAPI integrate with Pydantic for validation and OpenAPI generation?
**A:** When you define a path parameter or request body as a Pydantic model class, FastAPI automatically:
1. Parses the incoming JSON request payload.
2. Validates it against the Pydantic model's field rules. If validation fails, it automatically returns a standard `422 Unprocessable Entity` response with detail errors.
3. Serializes the response model back to JSON.
4. Reads the Pydantic field schemas to auto-generate the interactive Swagger UI (`/docs`) and Redoc docs, complete with request and response examples.

### Q4. How do you implement real-time token streaming (Server-Sent Events) in FastAPI?
**A:** To stream tokens as they are generated by the LLM, you use FastAPI's `StreamingResponse`. You write an asynchronous generator function that yields tokens one by one (usually wrapped in the Server-Sent Events format: `yield f"data: {token}\n\n"`). Then, you return this generator wrapped in `StreamingResponse` with the media type `text/event-stream`. On the frontend, client-side APIs (like `EventSource` or fetch-stream readers) read the stream line-by-line, displaying text dynamically to the user.

### Q5. How and when do you use FastAPI Background Tasks?
**A:** If an operation takes longer than ~500ms (like sending a notification email, logging metrics to an external service, or triggering a RAG re-indexing job), you shouldn't make the user wait for it. FastAPI provides `BackgroundTasks`. You add a `BackgroundTasks` parameter to your endpoint and call `background_tasks.add_task(my_func, arg1, arg2)`. FastAPI immediately returns the HTTP response to the user, and runs the registered task asynchronously in the background. For heavy, hours-long processing, use Celery or Redis Queue instead.

### Q6. How do you set up persistent WebSockets in FastAPI?
**A:** WebSockets allow for bidirectional, real-time communication between the client and server over a single TCP connection. In FastAPI, you declare an endpoint with `websocket: WebSocket` and write an async loop to handle connection approval, message receiving, and sending:
```python
@app.websocket("/chat")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            data = await websocket.receive_text()
            # process data and call LLM
            await websocket.send_text(f"Response: {response}")
    except WebSocketDisconnect:
        # cleanup code
```
This is the standard approach for building conversational chat interfaces where latency needs to be extremely low.

### Q7. How do you configure CORS in FastAPI? What is custom middleware?
**A:** CORS (Cross-Origin Resource Sharing) is a browser security mechanism. To allow frontends running on other domains (like localhost:3000) to fetch your API, you add the `CORSMiddleware` using `app.add_middleware()`, specifying allowed origins, methods, and headers.
Custom middleware is a component that wraps every incoming request and outgoing response. You define it using the `@app.middleware("http")` decorator. You can use it to log request latency, add custom headers, check API keys, or rate limit incoming calls before they hit your endpoint logic.

### Q8. How do you write unit and integration tests for async FastAPI endpoints?
**A:** We use `pytest` along with `httpx.AsyncClient` or FastAPI's `TestClient`.
- For sync endpoints, `TestClient` is sufficient.
- For async endpoints, we use `httpx.AsyncClient` inside a pytest async test function (annotated with `@pytest.mark.anyio` or using `pytest-asyncio`). We initialize the client with `AsyncClient(app=app, base_url="http://test")`, invoke path methods using `await client.get("/endpoint")`, and assert status codes and JSON outputs. We mock external LLM APIs using libraries like `pytest-mock` to avoid making real API calls during testing.

### Q9. How do you implement JWT-based authentication in FastAPI?
**A:** FastAPI provides a `security` module. We use `OAuth2PasswordBearer` to specify the token URL and extract the bearer token from the request's Authorization header:
```python
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")
```
We write a dependency function `get_current_user(token: str = Depends(oauth2_scheme))` that decodes the JWT using a library like `python-jose`, validates the signature, extracts the user ID, checks the database, and returns the user object. Any protected route simply adds `current_user: User = Depends(get_current_user)` to its parameters.

### Q10. What are lifespan events in FastAPI? How are they useful for AI workloads?
**A:** Lifespan events manage startup and shutdown processes. You define an `@asynccontextmanager` function that runs code *before* the application starts receiving requests (e.g., loading a sentence-transformer model into GPU memory, initializing a FAISS index, or opening a database connection pool), yields control to the app, and executes cleanup code *after* the application shuts down (e.g., closing connections, freeing up resources).
```python
@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup: Load model
    app.state.model = SentenceTransformer("all-MiniLM-L6-v2")
    yield
    # Shutdown: Clean resources
    del app.state.model
```
It prevents loading heavy models on every request, improving API latency.